# 학습 데이터 v2 — train 매칭 + 대량 추출

## 지난번과 뭐가 다른가

대회 train 16,373문제에는 **답만 있고 풀이가 없습니다.**
그런데 그중 상당수가 NuminaMath에 **풀이와 함께** 있습니다(평가 문항 기준 86% 일치를 이미 확인).

→ 문제 문장으로 대조해서 풀이를 붙이면, **대회 분포에 정확히 일치하는 고품질 학습 데이터**가 나옵니다.
지난번의 무작위 2만 개보다 표적화돼 있습니다.

| | 지난번 (8/18) | 이번 |
|---|---|---|
| 주 데이터 | NuminaMath 무작위 20k | **train 매칭분** (대회 문제 그 자체) |
| 보조 | RFT 5.4k | train 매칭분 + 무작위 추출 + RFT |
| 총량 | 25,465 | **약 30,000** |

## 규칙 대조 (진행 전 확인)

| 규칙 | 이 작업 | 판정 |
|---|---|---|
| 5.1a 주최 제공 학습 데이터 활용 | train을 주 데이터로 사용 | ○ 권장 사항 |
| 5.1b test.parquet 학습 사용 금지 | test는 손대지 않음 (파일도 없음) | ○ |
| 5.2a 공개 데이터셋 자유 | NuminaMath-1.5, Apache 2.0 | ○ |
| 5.2b 유료·비공개 데이터 금지 | 무료·동등 접근 | ○ |
| 5.2c 사용 목록 명시 | **제출 시 `AI-MO/NuminaMath-1.5` 기재 필요** | ▲ 잊지 말 것 |
| 5.3c 테스트 문제 답 조회 금지 | 대조 대상은 **train 문제뿐** | ○ |

**추가 안전장치**: 검증셋 300문제와 리더보드 831문제는 매칭·추출 양쪽에서 모두 제외합니다.
평가의 정직성을 위해 규칙이 요구하는 것보다 보수적으로 갑니다.

## 실행
GPU 불필요 (**Accelerator = None**), Internet On. 약 20분.
`[1]` `[2]` → ⛔Restart → `[1]` → `[3]`~`[8]`


---
## [1] 설정 ▶️

In [ ]:
N_EXTRA    = 12000     # train 매칭분에 더할 NuminaMath 무작위 추출 수
MAX_SOL_CH = 3000
MIN_SOL_CH = 100
VALID_N    = 300       # 추론 노트북과 동일 — 절대 변경 금지
SEED       = 42        # 추론 노트북과 동일 — 절대 변경 금지
OUT_PATH   = "/kaggle/working/train_v2.jsonl"
print("GPU 불필요 / Internet On")

---
## [2] 설치 ⏭️
---
## ⛔ Restart Session → [1]부터
---

In [ ]:
!pip install -q -U datasets 2>&1 | tail -2
import datasets; print("datasets", datasets.__version__)

---
## [3] 대회 데이터 + 차단 목록 ▶️

**차단 목록**을 먼저 만듭니다. 검증셋 300 + 리더보드 831.
매칭에서도, 무작위 추출에서도 이 문항들은 제외됩니다.

정규화(소문자화 → LaTeX 명령 제거 → 영숫자만 남김)한 문장으로 대조하므로
표기가 조금 달라도 같은 문제면 잡힙니다.

In [ ]:
import glob, os, re, pandas as pd

def find_csv(must_have, must_not=()):
    for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
        b = os.path.basename(p).lower()
        if all(k in b for k in must_have) and not any(k in b for k in must_not):
            return p

TRAIN_PATH = find_csv(["train"], must_not=["filtered","ids","leaderboard","test"])
BAD_PATH   = find_csv(["filtered","ids"])
LB_PATH    = find_csv(["leaderboard","filtered"]) or find_csv(["leaderboard"])
assert TRAIN_PATH and LB_PATH and TRAIN_PATH != BAD_PATH

train = pd.read_csv(TRAIN_PATH)
train = train[~train["id"].isin(set(pd.read_csv(BAD_PATH)["id"]))].reset_index(drop=True)
assert len(train) == 16373, f"16373이어야 하는데 {len(train)}"
lb = pd.read_csv(LB_PATH)

def norm(s):
    s = str(s).lower()
    s = re.sub(r"\\[a-z]+", " ", s)
    s = re.sub(r"[^a-z0-9]+", "", s)
    return s

valid = train.sample(VALID_N, random_state=SEED)
print("검증셋 첫 id:", valid.iloc[0]["id"], "(train-004925 여야 함)")

BLOCK = set(norm(q) for q in valid["question"]) | set(norm(q) for q in lb["question"])
BLOCK.discard("")

# 매칭 대상 = train에서 검증셋을 뺀 것
pool = train[~train["id"].isin(set(valid["id"]))].reset_index(drop=True)
pool["nq"] = pool["question"].map(norm)
print(f"\n매칭 대상 train: {len(pool):,}  (16373 - {VALID_N})")
print(f"차단 목록: {len(BLOCK):,}")

---
## [4] NuminaMath 다운로드 ▶️ 3~5분

In [ ]:
from datasets import load_dataset
nm = load_dataset("AI-MO/NuminaMath-1.5", split="train").to_pandas()
print(nm.shape)
print(nm["question_type"].value_counts().head())

---
## [5] 공통 필터 ▶️ (지난번과 동일)

| 조건 | 이유 |
|---|---|
| `math-word-problem` | 증명·객관식 제외. 대회는 숫자 답 |
| `problem_is_valid` / `solution_is_valid` = Yes | 데이터셋 자체 품질 표시 |
| 이미지 링크 없음 | 그림이 본문이면 텍스트만으론 못 품 |
| 정수 답 (int64 범위) | 대회는 정답이 항상 정수 |
| 풀이 100~3000자 | 너무 짧으면 풀이가 아니고, 길면 학습 비용만 증가 |

In [ ]:
import re
from collections import Counter

def extract_boxed(text):
    """Return the raw content inside the LAST \\boxed{...}, brace-balanced."""
    idx = text.rfind('\\boxed')
    if idx == -1:
        return None
    i = idx + len('\\boxed')
    while i < len(text) and text[i] == ' ':
        i += 1
    if i >= len(text):
        return None
    if text[i] != '{':                       # bare form: \boxed 15
        m = re.match(r'-?[\d,]+', text[i:])
        return m.group(0) if m else None
    depth, start = 0, i + 1
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None

def to_int(s):
    """LaTeX/text -> python int, or None. Never uses float(), so huge ints survive."""
    if s is None:
        return None
    s = str(s).strip()
    s = s.replace('{,}', '').replace('{\\,}', '')          # LaTeX thousands separator
    s = re.sub(r'\\(?:text|mathrm|mbox|textbf|textrm)\s*\{([^{}]*)\}', r'\1', s)
    for junk in ['\\!', '\\,', '\\;', '\\:', '\\ ', '\\left', '\\right',
                 '\\$', '$', '%', '~', '^\\circ', '\\%']:
        s = s.replace(junk, '')
    s = s.replace(',', '').replace(' ', '').strip()
    s = re.sub(r'[a-zA-Z]+$', '', s)                       # trailing unit: 42cm -> 42
    while len(s) > 1 and s[0] == '(' and s[-1] == ')':     # (\frac{100}{4}) -> \frac{100}{4}
        s = s[1:-1].strip()
    s = s.rstrip('.')
    if not s:
        return None
    m = re.fullmatch(r'\\[dt]?frac\{([-+]?\d+)\}\{([-+]?\d+)\}', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)/([-+]?\d+)', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)(?:\\times|\\cdot)10\^\{?(\d+)\}?', s)
    if m:
        return int(m.group(1)) * 10 ** int(m.group(2))
    if re.fullmatch(r'[-+]?\d+', s):
        return int(s)
    m = re.fullmatch(r'([-+]?\d+)\.0*', s)
    if m:
        return int(m.group(1))
    return None

def last_int(text):
    for c in reversed(re.findall(r'-?\d[\d,]*', text)):
        v = to_int(c)
        if v is not None:
            return v
    return None

def parse_answer(text):
    """None means 'this sample produced no usable integer' -> dropped from voting."""
    raw = extract_boxed(text)
    if raw is not None:
        return to_int(raw)          # boxed present but unparseable -> None, do NOT guess
    m = re.findall(r'(?:answer|Answer|ANSWER)\s*(?:is|:|=)+\s*\$?(-?[\d,]+)', text)
    if m:
        v = to_int(m[-1])
        if v is not None:
            return v
    return last_int(text)

def majority_vote(values, fallback=0):
    vals = [v for v in values if v is not None]
    if not vals:
        return fallback
    return Counter(vals).most_common(1)[0][0]


INT64_MAX = 2**63 - 1
def to_int_safe(s):
    v = to_int(s)
    return None if (v is None or abs(v) > INT64_MAX) else v

n0 = len(nm)
f = nm[nm["question_type"] == "math-word-problem"].copy()
f = f[(f["problem_is_valid"] == "Yes") & (f["solution_is_valid"] == "Yes")]
img = f["problem"].str.contains(r"!\[\]|cdn\.mathpix|\.jpg|\.png", regex=True, na=False) | \
      f["solution"].str.contains(r"!\[\]|cdn\.mathpix|\.jpg|\.png", regex=True, na=False)
f = f[~img]
f["ans_int"] = f["answer"].map(to_int_safe).astype("object")
f = f[f["ans_int"].notna()]
sl = f["solution"].str.len()
f = f[(sl >= MIN_SOL_CH) & (sl <= MAX_SOL_CH)]
f["nq"] = f["problem"].map(norm)
f = f.drop_duplicates(subset="nq")
print(f"NuminaMath {n0:,} → 필터 통과 {len(f):,}")

---
## [6] ★ train 매칭 ▶️ 이번의 핵심

정규화 문장으로 대회 train과 NuminaMath를 대조합니다.

### 안전장치: 답 일치 검사
매칭된 문제라도 **NuminaMath의 답과 대회의 답이 다르면 버립니다.**
문장은 같은데 답이 다르면 둘 중 하나가 잘못된 것이고, 그런 걸 학습하면 해롭습니다.

이 검사가 매칭 품질의 척도이기도 합니다 — 불일치가 많으면 매칭 로직을 의심해야 합니다.

In [ ]:
nm_by_q = f.set_index("nq")[["solution", "ans_int"]]
joined = pool.join(nm_by_q, on="nq", how="inner")
print(f"문장 매칭: {len(joined):,} / {len(pool):,}  ({len(joined)/len(pool):.1%})")

same = joined["answer"].astype("int64") == joined["ans_int"].astype("int64")
print(f"답까지 일치: {same.sum():,}  (불일치 {(~same).sum():,}개 제외)")
matched = joined[same].copy()

# 차단 목록 재확인 (이중 안전장치)
hit = matched["nq"].isin(BLOCK)
if hit.sum():
    print(f"▲ 차단 목록과 겹침 {hit.sum()}개 → 제거")
    matched = matched[~hit]
print(f"\n★ train 매칭 최종: {len(matched):,}개")

---
## [7] 무작위 추출 + RFT 혼합 ▶️

**우선순위**: train 매칭분 → RFT → NuminaMath 무작위 추출

무작위 추출분은 **이미 쓴 문항(매칭분)과 차단 목록을 모두 제외**하고,
`source`별 비율을 유지하며 뽑습니다(쉬운 문장제~올림피아드 균형).

In [ ]:
import json, random

records = []
for _, r in matched.iterrows():
    records.append({"question": str(r["question"]).strip(),
                    "solution": str(r["solution"]).strip(),
                    "answer": int(r["answer"]), "src": "train_match"})

used = set(matched["nq"])
rest = f[~f["nq"].isin(used | BLOCK)]
print(f"무작위 추출 후보: {len(rest):,}")

frac = min(1.0, N_EXTRA / len(rest))
extra = (rest.groupby("source", group_keys=False)
             .apply(lambda g: g.sample(max(1, int(round(len(g)*frac))), random_state=SEED)))
extra = extra.sample(min(N_EXTRA, len(extra)), random_state=SEED)

n_fix = 0
for _, r in extra.iterrows():
    sol, ans = str(r["solution"]).strip(), int(r["ans_int"])
    if parse_answer(sol) != ans:
        sol = sol.rstrip() + f"\n\nThe final answer is $\\boxed{{{ans}}}$."
        n_fix += 1
    records.append({"question": str(r["problem"]).strip(), "solution": sol,
                    "answer": ans, "src": "numina_random"})
print(f"무작위 추출 {len(extra):,}개 (boxed 보정 {n_fix:,})")

# train 매칭분도 boxed 보정
n_fix2 = 0
for r in records:
    if r["src"] == "train_match" and parse_answer(r["solution"]) != r["answer"]:
        r["solution"] = r["solution"].rstrip() + f"\n\nThe final answer is $\\boxed{{{r['answer']}}}$."
        n_fix2 += 1
print(f"train 매칭분 boxed 보정 {n_fix2:,}")

# RFT 추가
rp = next((p for p in glob.glob("/kaggle/input/**/*.jsonl", recursive=True)
           if "rft" in os.path.basename(p).lower()), None)
if rp:
    rft = [json.loads(l) for l in open(rp, encoding="utf-8")]
    easy = [r for r in rft if r["n_correct"] == 4]
    hard = [r for r in rft if r["n_correct"] < 4]
    random.Random(SEED).shuffle(easy)
    for r in hard + easy[:1800]:
        records.append({"question": r["question"], "solution": r["solution"],
                        "answer": int(r["answer"]), "src": "rft"})
    print(f"RFT {len(hard)+min(1800,len(easy)):,}개 추가")

random.Random(SEED).shuffle(records)
from collections import Counter
print(f"\n최종 {len(records):,}개  {dict(Counter(r['src'] for r in records))}")

---
## [8] 저장 ▶️

저장 후 **반드시 다운로드**하고 Kaggle Dataset(`train-v2-data`)으로 올리세요.
`/kaggle/working/`은 세션 종료 시 사라집니다.

In [ ]:
with open(OUT_PATH, "w", encoding="utf-8") as fp:
    for r in records:
        fp.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"저장: {OUT_PATH} ({os.path.getsize(OUT_PATH)/1e6:.1f} MB, {len(records):,}줄)")

ex = next(r for r in records if r["src"] == "train_match")
print("\n" + "="*70 + "\ntrain 매칭 샘플\n" + "="*70)
print("[문제]", ex["question"][:220])
print("\n[풀이]", ex["solution"][:450])
print("\n[정답]", ex["answer"])

print("\n" + "="*70)
print("최종 제출 시 명시할 외부 데이터셋 (규칙 5.2c)")
print("  AI-MO/NuminaMath-1.5  (Apache 2.0)")
print("="*70)

from IPython.display import FileLink
FileLink("train_v2.jsonl")